# MÓDULO 01 — Inventario de escenas Landsat 8/9

**Proyecto:** `lst-barranquilla-suhi-deep-learning`  
**Área:** Barranquilla, Colombia  
**Periodo:** 2013–2026  
**WRS:** Path 9 / Row 52  
**Colecciones GEE:**
- `LANDSAT/LC08/C02/T1_L2`
- `LANDSAT/LC09/C02/T1_L2`

## Objetivo

Construir el inventario tabular de escenas Landsat 8/9 Collection 2 Level-2 disponibles para el path/row de Barranquilla durante el periodo de análisis.

Este módulo no descarga imágenes, no modifica productos raster y no calcula índices espectrales. Su función es documentar la disponibilidad temporal de escenas y generar tablas CSV trazables para los módulos posteriores.

## Entradas

- Google Earth Engine.
- Landsat 8 Collection 2 Level-2.
- Landsat 9 Collection 2 Level-2.
- `WRS_PATH = 9`.
- `WRS_ROW = 52`.

## Salidas

Las salidas se guardan en:

```text
data/scene_inventory/
```

Archivos generados:

```text
scene_inventory_landsat89.csv
annual_scene_count_landsat89.csv
monthly_scene_count_landsat89.csv
```

## Nota de reproducibilidad

El identificador del proyecto de Earth Engine no se fija dentro del notebook. Cada usuario debe configurar su propio proyecto si su entorno lo requiere.


In [ ]:
# ============================================================
# CELDA 1 — Instalación opcional e imports
# ============================================================

import sys
import subprocess
from pathlib import Path

# En Google Colab, earthengine-api suele estar disponible.
# Si no lo está, se instala de forma silenciosa.
try:
    import ee
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "earthengine-api",
    ])
    import ee

import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)


In [ ]:
# ============================================================
# CELDA 2 — Directorio del repositorio y salidas
# ============================================================

# Este notebook está diseñado para ejecutarse desde la raíz del repositorio
# o desde la carpeta notebooks/.
#
# Si se ejecuta en Colab, puede montar Google Drive, pero las rutas personales
# no quedan fijadas en el código.

if IN_COLAB:
    drive.mount("/content/drive")

CURRENT_DIR = Path.cwd()

# Si el notebook se ejecuta desde /notebooks, subimos un nivel.
if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
else:
    REPO_DIR = CURRENT_DIR

OUT_DIR = REPO_DIR / "data" / "scene_inventory"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SCENE_CSV = OUT_DIR / "scene_inventory_landsat89.csv"
OUT_YEAR_CSV = OUT_DIR / "annual_scene_count_landsat89.csv"
OUT_MONTH_CSV = OUT_DIR / "monthly_scene_count_landsat89.csv"

print("CURRENT_DIR:", CURRENT_DIR)
print("REPO_DIR:", REPO_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
# ============================================================
# CELDA 3 — Inicialización de Google Earth Engine
# ============================================================

# Configure aquí su proyecto de Earth Engine si su cuenta lo requiere.
# Ejemplo:
# EE_PROJECT_ID = "your-earth-engine-project-id"
#
# Si no se requiere project ID explícito, deje:
EE_PROJECT_ID = None

try:
    if EE_PROJECT_ID:
        ee.Initialize(project=EE_PROJECT_ID)
    else:
        ee.Initialize()
    print("Earth Engine ya estaba inicializado.")
except Exception as e:
    print("Inicialización falló:")
    print(e)
    print("\nAutenticando Earth Engine en modo notebook...")

    ee.Authenticate(auth_mode="notebook")

    if EE_PROJECT_ID:
        ee.Initialize(project=EE_PROJECT_ID)
    else:
        ee.Initialize()

    print("Earth Engine inicializado correctamente.")

print(ee.String("Earth Engine OK").getInfo())


In [ ]:
# ============================================================
# CELDA 4 — Configuración del inventario Landsat 8/9
# ============================================================

START_YEAR = 2013
END_YEAR = 2026

START_DATE = f"{START_YEAR}-01-01"
END_DATE = f"{END_YEAR + 1}-01-01"

WRS_PATH = 9
WRS_ROW = 52

LANDSAT_COLLECTIONS = {
    "LC08": "LANDSAT/LC08/C02/T1_L2",
    "LC09": "LANDSAT/LC09/C02/T1_L2",
}

print("Periodo:", START_DATE, "a", END_DATE)
print("WRS_PATH:", WRS_PATH)
print("WRS_ROW:", WRS_ROW)
print("Colecciones:", LANDSAT_COLLECTIONS)


In [ ]:
# ============================================================
# CELDA 5 — Cargar colecciones Landsat 8/9 C2 L2 desde GEE
# ============================================================

def prepare_collection(collection_id: str, sensor_name: str) -> ee.ImageCollection:
    """Filtra una colección Landsat por fecha y WRS path/row."""
    return (
        ee.ImageCollection(collection_id)
        .filterDate(START_DATE, END_DATE)
        .filterMetadata('WRS_PATH', 'equals', WRS_PATH)
        .filterMetadata('WRS_ROW', 'equals', WRS_ROW)
        .map(lambda img: img.set('SENSOR', sensor_name))
    )

l8 = prepare_collection(LANDSAT_COLLECTIONS['LC08'], 'LC08')
l9 = prepare_collection(LANDSAT_COLLECTIONS['LC09'], 'LC09')

l89 = l8.merge(l9).sort('system:time_start')

n_scenes = int(l89.size().getInfo())
print('Total escenas encontradas:', n_scenes)

if n_scenes == 0:
    raise RuntimeError('No se encontraron escenas para la configuración indicada.')


In [ ]:
# ============================================================
# CELDA 6 — Extraer inventario de escenas
# ============================================================

def image_to_feature(img: ee.Image) -> ee.Feature:
    """Convierte metadatos relevantes de una imagen Landsat en Feature."""
    date = ee.Date(img.get('system:time_start'))

    return ee.Feature(None, {
        'scene_id': img.get('LANDSAT_PRODUCT_ID'),
        'sensor': img.get('SENSOR'),
        'date': date.format('YYYY-MM-dd'),
        'year': date.get('year'),
        'month': date.get('month'),
        'doy': date.getRelative('day', 'year').add(1),
        'wrs_path': img.get('WRS_PATH'),
        'wrs_row': img.get('WRS_ROW'),
        'cloud_cover': img.get('CLOUD_COVER'),
        'cloud_cover_land': img.get('CLOUD_COVER_LAND'),
        'collection_category': img.get('COLLECTION_CATEGORY'),
        'processing_level': img.get('PROCESSING_LEVEL'),
        'spacecraft_id': img.get('SPACECRAFT_ID'),
    })

features = l89.map(image_to_feature)
inventory = features.getInfo()['features']
rows = [f['properties'] for f in inventory]

df = pd.DataFrame(rows)

cols = [
    'scene_id',
    'sensor',
    'date',
    'year',
    'month',
    'doy',
    'wrs_path',
    'wrs_row',
    'cloud_cover',
    'cloud_cover_land',
    'collection_category',
    'processing_level',
    'spacecraft_id',
]

df = df[cols].sort_values(['year', 'date', 'sensor']).reset_index(drop=True)

df.to_csv(OUT_SCENE_CSV, index=False)

print('Inventario guardado en:')
print(OUT_SCENE_CSV)
print('Número de escenas:', len(df))

display(df.head(20))
display(df.tail(20))


In [ ]:
# ============================================================
# CELDA 7 — Resumen anual
# ============================================================

annual = (
    df.groupby('year')
    .agg(
        n_scenes=('scene_id', 'count'),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
        mean_cloud_cover=('cloud_cover', 'mean'),
        median_cloud_cover=('cloud_cover', 'median'),
        min_cloud_cover=('cloud_cover', 'min'),
        max_cloud_cover=('cloud_cover', 'max'),
        months=('month', lambda x: sorted(list(set(x)))),
    )
    .reset_index()
)

annual.to_csv(OUT_YEAR_CSV, index=False)

print('Resumen anual guardado en:')
print(OUT_YEAR_CSV)

display(annual)


In [ ]:
# ============================================================
# CELDA 8 — Resumen mensual
# ============================================================

monthly = (
    df.groupby(['year', 'month'])
    .agg(
        n_scenes=('scene_id', 'count'),
        mean_cloud_cover=('cloud_cover', 'mean'),
        median_cloud_cover=('cloud_cover', 'median'),
        scene_ids=('scene_id', lambda x: ' | '.join(x)),
    )
    .reset_index()
    .sort_values(['year', 'month'])
)

monthly.to_csv(OUT_MONTH_CSV, index=False)

print('Resumen mensual guardado en:')
print(OUT_MONTH_CSV)

display(monthly)


In [ ]:
# ============================================================
# CELDA 9 — Verificación final de salidas
# ============================================================

expected_outputs = [OUT_SCENE_CSV, OUT_YEAR_CSV, OUT_MONTH_CSV]

for path in expected_outputs:
    if not path.exists():
        raise FileNotFoundError(f'No se generó el archivo esperado: {path}')
    if path.stat().st_size == 0:
        raise RuntimeError(f'Archivo generado vacío: {path}')

print('\n==============================')
print('MÓDULO 01 COMPLETADO')
print('==============================')

print('\nResumen anual rápido:')
for _, r in annual.iterrows():
    print(
        f"{int(r['year'])}: "
        f"{int(r['n_scenes'])} escenas | "
        f"meses={r['months']} | "
        f"cloud median={r['median_cloud_cover']:.1f}%"
    )

print('\nArchivos generados:')
for path in expected_outputs:
    print(path)
